# Verification Agent

Checks if proposed resolution steps are **complete**, **safe**, and **aligned** with documentation.

> L3 severity issues are always flagged for human review without calling the LLM.

## Step 1 — Install Dependencies

Run once to install all required packages.

In [ ]:
import subprocess, sys

packages = [
    "langchain>=0.3.0",
    "langchain-core>=0.3.0",
    "langchain-google-genai>=2.0.0",
    "google-generativeai>=0.8.0",
    "python-dotenv>=1.0.0",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed successfully.")


## Step 2 — Load Environment Variables

Loads `GOOGLE_API_KEY` from the `.env` file in the project root.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# agents/ directory -> project root is one level up
env_path = Path("../.env")
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"Loaded .env from: {env_path.resolve()}")
else:
    print("WARNING: .env not found — set GOOGLE_API_KEY manually below")
    # Uncomment and set your key if .env is missing:
    # os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"

api_key = os.getenv("GOOGLE_API_KEY", "")
if api_key:
    masked = "*" * (len(api_key) - 6) + api_key[-6:]
    print(f"GOOGLE_API_KEY found: {masked}")
else:
    print("ERROR: GOOGLE_API_KEY is not set — LLM calls will fail!")


## Step 3 — Define AgentState Schema

Self-contained TypedDict that mirrors `agents/state.py` — no relative import needed.

In [ ]:
from typing import TypedDict, List


class AgentState(TypedDict):
    """Shared state schema for LangGraph agent pipeline."""

    # Input
    user_query: str

    # Classification
    category: str          # e.g. "deployment", "database", "cpu", "network", "general"
    severity: str          # "L1", "L2", "L3"
    summary: str           # One-line summary of the issue

    # Knowledge base results
    kb_results: List[dict] # [{text, source, score}]
    kb_context: str        # Formatted context string for LLM

    # Diagnostic output
    root_cause: str
    confidence: str        # "high", "medium", "low"

    # Resolution
    resolution_steps: str
    resolution_status: str # "resolved", "escalated", "needs_human"

    # Verification
    verified: bool
    verification_notes: str

    # Escalation
    escalated: bool
    escalation_reason: str

    # Final answer shown to user
    final_answer: str


print("AgentState schema defined.")


## Step 4 — Define the Verification Agent

- **L3 / needs_human** → skip LLM, mark `verified = False` immediately  
- **All other severities** → ask Gemini to evaluate completeness & safety

In [ ]:
import json
import re

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage


SYSTEM_PROMPT = """You are a senior SRE reviewing a proposed resolution for a support ticket.
Check if the resolution steps are:
1. Complete (cover the root cause fully)
2. Safe (no risky undocumented actions)
3. Aligned with the documentation provided

Return ONLY a valid JSON object:
{
  "verified": <true | false>,
  "notes": "<brief verification notes or concerns>"
}"""


def verification_agent(state: AgentState) -> AgentState:
    """
    Verification Agent - checks if the resolution steps are complete and safe.
    For L3 issues, flags for human review without calling the LLM.
    """
    # L3 always needs human review - mark as not verified immediately
    if state["severity"] == "L3" or state.get("resolution_status") == "needs_human":
        state["verified"] = False
        state["verification_notes"] = (
            "L3 severity or complex resolution - requires human engineer review before execution."
        )
        return state

    # Build the LLM client
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

    user_prompt = (
        f"Original Issue: {state['user_query']}\n"
        f"Root Cause: {state['root_cause']}\n\n"
        f"Proposed Resolution:\n{state['resolution_steps']}\n\n"
        f"Documentation Context:\n{state.get('kb_context', 'No context available.')}\n\n"
        "Is this resolution complete and safe?"
    )

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ]

    response = llm.invoke(messages)
    raw = response.content.strip()

    # Strip markdown code fences if the LLM wraps its JSON in them
    raw = re.sub(r"^```[a-z]*\n?", "", raw)
    raw = re.sub(r"\n?```$", "", raw)

    try:
        data = json.loads(raw)
        state["verified"] = bool(data.get("verified", False))
        state["verification_notes"] = data.get("notes", "")
    except json.JSONDecodeError:
        # Fallback: auto-verify if LLM response could not be parsed
        state["verified"] = True
        state["verification_notes"] = "Auto-verified (parse error in LLM response)"

    return state


print("verification_agent() defined successfully.")


## Step 5 — Test the Agent

### 5a — L3 Severity (human review flag, no LLM call)

In [ ]:
sample_state_l3: AgentState = {
    "user_query": "Production database is completely unresponsive after migration.",
    "category": "database",
    "severity": "L3",
    "summary": "DB unresponsive post-migration",
    "kb_results": [],
    "kb_context": "",
    "root_cause": "Schema migration left orphaned locks on critical tables.",
    "confidence": "high",
    "resolution_steps": "Kill all active sessions and roll back the migration.",
    "resolution_status": "needs_human",
    "verified": False,
    "verification_notes": "",
    "escalated": False,
    "escalation_reason": "",
    "final_answer": "",
}

result_l3 = verification_agent(sample_state_l3)

print("=== L3 Verification Result ===")
print(f"Verified : {result_l3['verified']}")
print(f"Notes    : {result_l3['verification_notes']}")


### 5b — L1 Severity (live Gemini verification call)

In [ ]:
sample_state_l1: AgentState = {
    "user_query": "CPU usage spiked to 95% on app-server-01 during business hours.",
    "category": "cpu",
    "severity": "L1",
    "summary": "High CPU usage on app-server-01",
    "kb_results": [],
    "kb_context": (
        "Standard procedure: identify top processes with `top`, "
        "kill runaway processes, restart service if needed."
    ),
    "root_cause": "Runaway background job consuming all CPU cycles.",
    "confidence": "high",
    "resolution_steps": (
        "1. SSH into app-server-01.\n"
        "2. Run `top` to identify the high-CPU process.\n"
        "3. Kill the process with `kill -9 <PID>`.\n"
        "4. Monitor CPU for 5 minutes to confirm resolution.\n"
        "5. Restart the background job service if needed."
    ),
    "resolution_status": "resolved",
    "verified": False,
    "verification_notes": "",
    "escalated": False,
    "escalation_reason": "",
    "final_answer": "",
}

result_l1 = verification_agent(sample_state_l1)

print("=== L1 Verification Result ===")
print(f"Verified : {result_l1['verified']}")
print(f"Notes    : {result_l1['verification_notes']}")
